# S02E04 — Mailbox

**Zadanie:** Przeszukać skrzynkę pocztową (zmail API) i wydobyć trzy informacje:
1. `date` — kiedy zaplanowany jest atak (YYYY-MM-DD)
2. `password` — hasło pracownicze
3. `confirmation_code` — kod potwierdzenia (SEC- + 32 znaki hex = 36 łącznie)

**Techniki:** ReAct agent loop z GPT-4o, function calling (OpenAI tools), paginacja mailboxu

In [ ]:
import os
import json
import requests
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
AI_DEVS_KEY = os.environ["AI_DEVS_API_KEY"]

ZMAIL_URL  = "https://hub.ag3nts.org/api/zmail"
VERIFY_URL = "https://hub.ag3nts.org/verify"

print("Konfiguracja OK.")

In [ ]:
# ── API WRAPPERS ──────────────────────────────────────────────────────────────

def search_mail(query: str, page: int = 1) -> dict:
    """Szukaj maili operatorem Gmail: from:, to:, subject:, OR, AND."""
    resp = requests.post(ZMAIL_URL, json={
        "apikey": AI_DEVS_KEY, "action": "search", "query": query, "page": page,
    })
    return resp.json()


def get_mail(mail_id: str) -> dict:
    """Pobierz pełną treść maila po ID."""
    resp = requests.post(ZMAIL_URL, json={
        "apikey": AI_DEVS_KEY, "action": "getMessages", "ids": mail_id,
    })
    return resp.json()


def get_inbox(page: int = 1) -> dict:
    """Lista maili (od najnowszego). Użyj gdy search nic nie znajdzie."""
    resp = requests.post(ZMAIL_URL, json={
        "apikey": AI_DEVS_KEY, "action": "getInbox", "page": page,
    })
    return resp.json()


def submit_answer(date: str, password: str, confirmation_code: str) -> dict:
    """Wyślij trzy zebrane wartości do hub-u. Zwróci flagę lub błąd."""
    payload = {
        "apikey": AI_DEVS_KEY,
        "task": "mailbox",
        "answer": {"date": date, "password": password, "confirmation_code": confirmation_code},
    }
    resp = requests.post(VERIFY_URL, json=payload)
    print(f"  HTTP {resp.status_code}: {resp.text[:400]}")
    try:
        return resp.json()
    except Exception:
        return {"raw": resp.text}


# ── DISPATCHER ────────────────────────────────────────────────────────────────

def handle_tool_call(name: str, args: dict) -> str:
    print(f"\n  TOOL: {name}({args})")
    if name == "search_mail":
        result = search_mail(**args)
    elif name == "get_mail":
        result = get_mail(**args)
    elif name == "get_inbox":
        result = get_inbox(**args)
    elif name == "submit_answer":
        result = submit_answer(**args)
    else:
        result = {"error": f"Unknown tool: {name}"}
    print(f"  RESULT: {json.dumps(result, ensure_ascii=False)[:300]}")
    return json.dumps(result)


print("API wrappers gotowe.")

In [ ]:
# ── TOOL SCHEMAS ──────────────────────────────────────────────────────────────

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_mail",
            "description": (
                "Szukaj w skrzynce operatorem Gmail: from:, to:, subject:, OR, AND. "
                "Zwraca listę maili z metadanymi (bez treści). "
                "Przykład: 'from:proton.me'"
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Zapytanie z operatorami Gmail"},
                    "page": {"type": "integer", "description": "Numer strony, domyślnie 1"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_mail",
            "description": "Pobierz pełną treść maila po ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "mail_id": {"type": "string", "description": "ID maila z wyników search"},
                },
                "required": ["mail_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_inbox",
            "description": "Lista ostatnich maili ze skrzynki (paginacja, od najnowszego).",
            "parameters": {
                "type": "object",
                "properties": {
                    "page": {"type": "integer", "description": "Numer strony, domyślnie 1"},
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "submit_answer",
            "description": (
                "Wyślij zebrane wartości do weryfikacji. "
                "Hub zwróci flagę jeśli wszystko poprawne. "
                "Wywołaj gdy masz kandydatów na wszystkie trzy pola."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {"type": "string", "description": "Data ataku, format YYYY-MM-DD"},
                    "password": {"type": "string", "description": "Hasło pracownicze z mailboxa"},
                    "confirmation_code": {
                        "type": "string",
                        "description": "Kod potwierdzenia: SEC- + dokładnie 32 znaki hex = 36 łącznie",
                    },
                },
                "required": ["date", "password", "confirmation_code"],
            },
        },
    },
]


# ── SYSTEM PROMPT ─────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are an intelligence agent tasked with extracting three pieces of information
from a live email inbox via API. The mailbox is in Polish.

Your goal is to find:
1. date - when the security department plans to attack our power plant (format YYYY-MM-DD)
2. password - an employee system password that is somewhere in the mailbox
3. confirmation_code - a security ticket confirmation code (format: SEC- + 32 hex chars = 36 total)

Key facts:
- Wiktor (the informant who betrayed us) sent an email from a @proton.me address
- The mailbox is active — new emails may arrive during your search
- All emails are in Polish — search using Polish keywords
- Always fetch the full mail body after finding a match — never guess from subject alone
- The inbox has many pages — scan ALL pages if search fails

Strategy:
1. search from:proton.me → find Wiktor's email and read its thread for the attack date
2. Search Polish keywords for password: "hasło", "nowe hasło", "system pracowniczy"
3. Search "SEC-" for the confirmation code
4. If search fails, use get_inbox and iterate through ALL pages reading subjects
5. Once you have all three values, call submit_answer — it returns hub feedback on wrong values
6. Keep searching until you receive a flag {FLG:...} in the response

Important:
- Never give up — if search finds nothing, browse inbox page by page (there are up to 15 pages)
- Before submitting, count the confirmation_code chars: SEC- (4) + exactly 32 hex chars = 36 total
- If submit returns an error, re-read the source email and double-check each character
"""

print("Tools i system prompt gotowe.")

In [ ]:
# ── AGENT LOOP ────────────────────────────────────────────────────────────────

def run_agent():
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Start searching the mailbox and find all three values. Go."},
    ]

    print("=" * 60)
    print("  MAILBOX AGENT STARTING")
    print("=" * 60)

    for step in range(1, 31):
        print(f"\n{'─' * 60}")
        print(f"  STEP {step}")
        print(f"{'─' * 60}")

        response = client.chat.completions.create(
            model="gpt-4o",
            tools=TOOLS,
            messages=messages,
            tool_choice="auto",
        )

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print(f"\n  Agent: {msg.content}")
            messages.append({
                "role": "user",
                "content": (
                    "Don't stop. Keep searching and trying. "
                    "If you got an error from submit_answer, re-read the emails and verify each value carefully — "
                    "especially the confirmation_code (must be exactly SEC- + 32 hex chars = 36 chars total). "
                    "Try different search terms or browse inbox pages. Never give up until you see {FLG:...} in a response."
                ),
            })
            continue

        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = handle_tool_call(name, args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

            # Sprawdź flagę w odpowiedzi
            try:
                data = json.loads(result)
                if isinstance(data, dict):
                    for v in data.values():
                        if isinstance(v, str) and v.startswith("{FLG:"):
                            print(f"\n{'=' * 60}")
                            print(f"  FLAG FOUND: {v}")
                            print(f"{'=' * 60}")
                            return
            except Exception:
                pass

    print("\nStep limit reached without finding the flag.")


run_agent()